# Turn CUAD Dataset from JSON to DataFrame


In [19]:
import json
from pathlib import Path

import pandas as pd

DATA_DIR = Path('..') / 'data' / 'cuad'

In [20]:
def load_cuad(path):
    with open(path, encoding='utf-8') as f:
        return json.load(f)

full = load_cuad(DATA_DIR / 'CUADv1.json')
train = load_cuad(DATA_DIR / 'train_separate_questions.json')
test = load_cuad(DATA_DIR / 'test.json')

print(f"full  : version={full['version']}, contracts={len(full['data'])}")
print(f"train : version={train['version']}, contracts={len(train['data'])}")
print(f"test  : version={test['version']}, contracts={len(test['data'])}")

full  : version=aok_v1.0, contracts=510
train : version=aok_v1.0, contracts=408
test  : version=aok_v1.0, contracts=102


In [21]:
def to_dataframe(cuad, split_name):
    rows = []
    for doc in cuad['data']:
        title = doc['title']
        for para in doc['paragraphs']:
            context_len = len(para['context'])
            for qa in para['qas']:
                category = qa['id'].split('__', 1)[1] if '__' in qa['id'] else None
                answers = qa.get('answers', [])
                rows.append({
                    'split': split_name,
                    'contract_title': title,
                    'category': category,
                    'question': qa['question'],
                    'qa_id': qa['id'],
                    'is_impossible': qa.get('is_impossible', False),
                    'num_answers': len(answers),
                    'answer_texts': [a['text'] for a in answers],
                    'answer_starts': [a['answer_start'] for a in answers],
                    'context_length': context_len,
                })
    return pd.DataFrame(rows)

df_full = to_dataframe(full, 'full')
df_train = to_dataframe(train, 'train')
df_test = to_dataframe(test, 'test')

df_full.shape, df_train.shape, df_test.shape

((20910, 10), (22450, 10), (4182, 10))

In [22]:
df_full.head()

,split,contract_title,category,question,qa_id,is_impossible,num_answers,answer_texts,answer_starts,context_length
0,full,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,Document Name,Highlight the parts (if any) of this contract ...,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,False,1,[DISTRIBUTOR AGREEMENT],[44],54290
1,full,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,Parties,Highlight the parts (if any) of this contract ...,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,False,5,"[Distributor, Electric City Corp., Electric Ci...","[244, 148, 49574, 197, 212]",54290
2,full,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,Agreement Date,Highlight the parts (if any) of this contract ...,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,False,1,"[7th day of September, 1999.]",[263],54290
3,full,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,Effective Date,Highlight the parts (if any) of this contract ...,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,False,2,[The term of this Agreement shall be ten (10...,"[5268, 31058]",54290
4,full,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,Expiration Date,Highlight the parts (if any) of this contract ...,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,False,1,[The term of this Agreement shall be ten (10...,[5268],54290


In [23]:
df_train.head()

,split,contract_title,category,question,qa_id,is_impossible,num_answers,answer_texts,answer_starts,context_length
0,train,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,Document Name_0,Highlight the parts (if any) of this contract ...,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,False,1,[DISTRIBUTOR AGREEMENT],[44],54290
1,train,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,Parties_0,Highlight the parts (if any) of this contract ...,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,False,1,[Distributor],[244],54290
2,train,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,Parties_1,Highlight the parts (if any) of this contract ...,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,False,1,[Electric City of Illinois L.L.C.],[49574],54290
3,train,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,Parties_2,Highlight the parts (if any) of this contract ...,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,False,1,[Electric City of Illinois LLC],[212],54290
4,train,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,Parties_3,Highlight the parts (if any) of this contract ...,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,False,1,[Company],[197],54290


In [24]:
df_test.head()

,split,contract_title,category,question,qa_id,is_impossible,num_answers,answer_texts,answer_starts,context_length
0,test,LohaCompanyltd_20191209_F-1_EX-10.16_11917878_...,Document Name,Highlight the parts (if any) of this contract ...,LohaCompanyltd_20191209_F-1_EX-10.16_11917878_...,False,1,[SUPPLY CONTRACT],[14],11475
1,test,LohaCompanyltd_20191209_F-1_EX-10.16_11917878_...,Parties,Highlight the parts (if any) of this contract ...,LohaCompanyltd_20191209_F-1_EX-10.16_11917878_...,False,2,"[The seller:, The buyer/End-User: Shenzhen LOH...","[143, 49]",11475
2,test,LohaCompanyltd_20191209_F-1_EX-10.16_11917878_...,Agreement Date,Highlight the parts (if any) of this contract ...,LohaCompanyltd_20191209_F-1_EX-10.16_11917878_...,True,0,[],[],11475
3,test,LohaCompanyltd_20191209_F-1_EX-10.16_11917878_...,Effective Date,Highlight the parts (if any) of this contract ...,LohaCompanyltd_20191209_F-1_EX-10.16_11917878_...,True,0,[],[],11475
4,test,LohaCompanyltd_20191209_F-1_EX-10.16_11917878_...,Expiration Date,Highlight the parts (if any) of this contract ...,LohaCompanyltd_20191209_F-1_EX-10.16_11917878_...,False,1,"[The Contract is valid for 5 years, beginning ...",[10985],11475


In [25]:
def normalize_category(cat):
    if not cat or '_' not in cat:
        return cat
    head, _, tail = cat.rpartition('_')
    return head if tail.isdigit() else cat

for df in (df_full, df_train, df_test):
    df['category'] = df['category'].map(normalize_category)

def collapse_spans(df):
    def _agg(g):
        texts = [t for lst in g['answer_texts'] for t in lst]
        starts = [s for lst in g['answer_starts'] for s in lst]
        qid = g['qa_id'].iloc[0]
        base_qid = qid.rsplit('_', 1)[0] if qid.rsplit('_', 1)[-1].isdigit() else qid
        return pd.Series({
            'split': g['split'].iloc[0],
            'question': g['question'].iloc[0],
            'qa_id': base_qid,
            'is_impossible': bool(g['is_impossible'].all()),
            'num_answers': len(texts),
            'answer_texts': texts,
            'answer_starts': starts,
            'context_length': g['context_length'].iloc[0],
        })
    return (df.groupby(['contract_title', 'category'], as_index=False)
              .apply(_agg, include_groups=False)
              .reset_index(drop=True))

df_full = collapse_spans(df_full)
df_train = collapse_spans(df_train)
df_test = collapse_spans(df_test)

print('After normalisation:')
print(f'  full : {df_full.shape}')
print(f'  train: {df_train.shape}')
print(f'  test : {df_test.shape}')
print(f'  categories identical? {list(df_train.category.unique()) == list(df_test.category.unique()) == list(df_full.category.unique())}')
df_train.head()

After normalisation:
  full : (20910, 10)
  train: (16728, 10)
  test : (4182, 10)
  categories identical? True


,contract_title,category,split,question,qa_id,is_impossible,num_answers,answer_texts,answer_starts,context_length
0,2ThemartComInc_19990826_10-12G_EX-10.10_670028...,Affiliate License-Licensee,train,Highlight the parts (if any) of this contract ...,2ThemartComInc_19990826_10-12G_EX-10.10_670028...,True,0,[],[],29454
1,2ThemartComInc_19990826_10-12G_EX-10.10_670028...,Affiliate License-Licensor,train,Highlight the parts (if any) of this contract ...,2ThemartComInc_19990826_10-12G_EX-10.10_670028...,True,0,[],[],29454
2,2ThemartComInc_19990826_10-12G_EX-10.10_670028...,Agreement Date,train,Highlight the parts (if any) of this contract ...,2ThemartComInc_19990826_10-12G_EX-10.10_670028...,False,1,"[June 21, 1999]",[114],29454
3,2ThemartComInc_19990826_10-12G_EX-10.10_670028...,Anti-Assignment,train,Highlight the parts (if any) of this contract ...,2ThemartComInc_19990826_10-12G_EX-10.10_670028...,False,2,"[transferable or assignable., All rights (unde...","[12412, 12261]",29454
4,2ThemartComInc_19990826_10-12G_EX-10.10_670028...,Audit Rights,train,Highlight the parts (if any) of this contract ...,2ThemartComInc_19990826_10-12G_EX-10.10_670028...,False,3,"[Once every twelve (12) months, 2TheMart throu...","[8593, 8958, 8701]",29454


In [26]:
# Save the three dataframes as pickle (preserves list-valued columns
# `answer_texts` / `answer_starts` without extra dependencies).
OUT_DIR = DATA_DIR / 'cleaned'
OUT_DIR.mkdir(exist_ok=True)

df_full.to_pickle(OUT_DIR / 'full.pkl')
df_train.to_pickle(OUT_DIR / 'train.pkl')
df_test.to_pickle(OUT_DIR / 'test.pkl')

for p in sorted(OUT_DIR.glob('*.pkl')):
    print(f'  {p.name:<12}  {p.stat().st_size / 1024:>8.1f} KB')

  full.pkl       11759.2 KB
  test.pkl        2271.7 KB
  train.pkl       9452.8 KB
